In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
import joblib
import os

# 1. Load the clean dataset
print("--- ArthSetu Model Training Pipeline ---")
print("Loading clean data...")
# Adjust path if necessary to match your local folder structure
data_path = '../data/processed/clean_features.csv'

if not os.path.exists(data_path):
    print(f"❌ ERROR: Could not find {data_path}")
    print("Ensure you have run your cleaning notebook first.")
else:
    df = pd.read_csv(data_path)

    # 2. Split Features and Target
    # 'Default' is our target (1 for default, 0 for paid back)
    X = df.drop('Default', axis=1)
    y = df['Default']

    # 3. Train/Test Split
    # We use 'stratify=y' to ensure both sets have the same percentage of defaults
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 4. Balancing classes with SMOTE
    # This prevents the AI from being biased toward "No Default"
    print("Balancing classes with SMOTE...")
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

    # 5. Training the XGBoost Engine
    print("Training XGBoost Engine...")
    model = XGBClassifier(
        n_estimators=100, 
        max_depth=4, 
        learning_rate=0.1,
        eval_metric='logloss',
        use_label_encoder=False,
        n_jobs=-1,
        random_state=42
    )
    model.fit(X_train_balanced, y_train_balanced)

    # 6. Export the Model Artifact
    artifact_path = '../backend/models/arthsetu_xgb.pkl'
    # Ensure directory exists
    os.makedirs(os.path.dirname(artifact_path), exist_ok=True)
    
    joblib.dump(model, artifact_path)
    print(f"✅ SUCCESS: Model exported to {artifact_path}")

    # ==========================================
    # 📊 MENTOR FIX: MODEL EVALUATION & ACCURACY
    # ==========================================
    print("\n" + "="*40)
    print("       UNDERWRITING PERFORMANCE REPORT")
    print("="*40)

    # A. Predict classes (0 or 1)
    y_pred = model.predict(X_test)

    # B. Predict probabilities (Needed for ROC-AUC)
    y_pred_proba = model.predict_proba(X_test)[:, 1] 

    # C. Calculate Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Model Accuracy: {accuracy * 100:.2f}%")

    # D. Calculate ROC-AUC (The true measure of a credit model)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"ROC-AUC Score:  {roc_auc:.4f}")

    # E. Detailed Report
    print("\nDetailed Metrics:")
    print(classification_report(y_test, y_pred, target_names=["Good Standing", "Default"]))

    print("\nConfusion Matrix (Actual vs Predicted):")
    cm = confusion_matrix(y_test, y_pred)
    print(f"True Negatives:  {cm[0][0]} (Correctly predicted Paybacks)")
    print(f"False Positives: {cm[0][1]} (Wrongly predicted Defaults)")
    print(f"False Negatives: {cm[1][0]} (Wrongly predicted Paybacks)")
    print(f"True Positives:  {cm[1][1]} (Correctly predicted Defaults)")
    print("="*40)

--- ArthSetu Model Training Pipeline ---
Loading clean data...
Balancing classes with SMOTE...
Training XGBoost Engine...


/Users/bijaya/arthsetu_project/backend/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [02:08:25] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ SUCCESS: Model exported to ../backend/models/arthsetu_xgb.pkl

       UNDERWRITING PERFORMANCE REPORT
Model Accuracy: 92.12%
ROC-AUC Score:  0.8319

Detailed Metrics:
               precision    recall  f1-score   support

Good Standing       0.96      0.96      0.96     27995
      Default       0.41      0.40      0.40      2005

     accuracy                           0.92     30000
    macro avg       0.68      0.68      0.68     30000
 weighted avg       0.92      0.92      0.92     30000


Confusion Matrix (Actual vs Predicted):
True Negatives:  26835 (Correctly predicted Paybacks)
False Positives: 1160 (Wrongly predicted Defaults)
False Negatives: 1205 (Wrongly predicted Paybacks)
True Positives:  800 (Correctly predicted Defaults)
